<a href="https://colab.research.google.com/github/VirunaVidaswin/Final-Year-Project/blob/data-preprocessing/MAST_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

df = pd.read_json("hf://datasets/mcemri/MAST-Data/MAD_full_dataset.json")

In [8]:
# @title
import json
import pandas as pd

# 1. Quick look at the available keys/columns
print("Columns:", df.columns.tolist())

# 2. Inspect the structure of the first trace
sample_trace = df.iloc[0]
print("\nTop-level keys in a trace:", sample_trace.keys())

# Check how failure modes and messages are stored
# Typically stored under keys like 'trajectory', 'messages', 'turns', or 'annotations'
for col in df.columns:
    val = df[col].iloc[0]
    print(f"Column '{col}' type: {type(val)}")
    if isinstance(val, list) and len(val) > 0:
        print(f"  -> Sample element in '{col}': {val[0]}")
    elif isinstance(val, dict):
        print(f"  -> Sample keys in '{col}': {list(val.keys())}")

Columns: ['mas_name', 'llm_name', 'benchmark_name', 'trace_id', 'trace', 'mast_annotation']

Top-level keys in a trace: Index(['mas_name', 'llm_name', 'benchmark_name', 'trace_id', 'trace',
       'mast_annotation'],
      dtype='object')
Column 'mas_name' type: <class 'str'>
Column 'llm_name' type: <class 'str'>
Column 'benchmark_name' type: <class 'str'>
Column 'trace_id' type: <class 'numpy.int64'>
Column 'trace' type: <class 'dict'>
  -> Sample keys in 'trace': ['key', 'index', 'trajectory']
Column 'mast_annotation' type: <class 'dict'>
  -> Sample keys in 'mast_annotation': ['1.1', '1.2', '1.3', '1.4', '1.5', '2.1', '2.2', '2.3', '2.4', '2.5', '2.6', '3.1', '3.2', '3.3']


Reading Turns and Annotations

What it does: Checks the target failure labels ('2.3', '2.4', '2.6') in the first trace, and attempts to iterate over the trajectory field assuming it is a Python list of turns.

What it revealed:
The keys directly mirror the MAST taxonomy (FM-2.3, FM-2.4, FM-2.6).
Total turns in sample: 345536 and Turn 0: '['. This proved that trajectory was not a list of turns, but a massive serialized string (over 345,000 characters) containing raw terminal logs.

In [9]:
sample_trace = df.iloc[0]

print("=" * 50)
print("1. MAST ANNOTATION STRUCTURE:")
print("=" * 50)
for k in ["2.3", "2.4", "2.6"]:
    print(f"Key '{k}':", sample_trace["mast_annotation"].get(k))

print("\n" + "=" * 50)
print("2. TRAJECTORY STRUCTURE (First 2 turns):")
print("=" * 50)
trajectory = sample_trace["trace"].get("trajectory", [])
print(f"Total turns in sample: {len(trajectory)}")
for i, turn in enumerate(trajectory[:2]):
    print(f"\n--- Turn {i} ---")
    if isinstance(turn, dict):
        for k, v in turn.items():
            # Truncate long strings for clean display
            val_str = str(v)[:120] + "..." if len(str(v)) > 120 else str(v)
            print(f"  {k}: {val_str}")
    else:
        print(f"  Raw turn: {str(turn)[:150]}...")

1. MAST ANNOTATION STRUCTURE:
Key '2.3': 0
Key '2.4': 0
Key '2.6': 0

2. TRAJECTORY STRUCTURE (First 2 turns):
Total turns in sample: 345536

--- Turn 0 ---
  Raw turn: [...

--- Turn 1 ---
  Raw turn: 2...


What it does: Iterates over all 1,642 traces to count how many contain FM-2.3 (Task Derailment), FM-2.4 (Information Withholding), FM-2.6 (Reasoning-Action Mismatch), and completely failure-free (Clean) traces.


What it revealed:
Total: 1,642 traces (exact match with Cemri et al. 2025).
FM-2.3: 353 traces (21.5%)
FM-2.4: 24 traces (1.46%) — confirming the rare, fatal nature of Information Withholding (
n
≈
14
–
24
n≈14–24
).
FM-2.6: 610 traces (37.15%)
Clean/Safe: 405 traces (24.67%) — your negative baseline set.

In [10]:
def check_mode(annotation, mode_key):
    val = annotation.get(mode_key, False)
    # Check if it's boolean True, non-empty list, non-zero int, or truthy dict
    if isinstance(val, bool):
        return val
    if isinstance(val, (list, dict)):
        return len(val) > 0
    if isinstance(val, (int, float)):
        return val > 0
    return False


audit = {"FM-2.3": 0, "FM-2.4": 0, "FM-2.6": 0, "Clean_No_Failures": 0}

for _, row in df.iterrows():
    ann = row["mast_annotation"]

    has_23 = check_mode(ann, "2.3")
    has_24 = check_mode(ann, "2.4")
    has_26 = check_mode(ann, "2.6")

    if has_23:
        audit["FM-2.3"] += 1
    if has_24:
        audit["FM-2.4"] += 1
    if has_26:
        audit["FM-2.6"] += 1

    # Check if entire trace has zero failures across all 14 modes
    has_any_failure = any(check_mode(ann, k) for k in ann.keys())
    if not has_any_failure:
        audit["Clean_No_Failures"] += 1

total = len(df)
print(f"Total Traces: {total:,}\n")
for mode, count in audit.items():
    print(f"{mode:18}: {count:5} traces ({count / total * 100:.2f}%)")

Total Traces: 1,642

FM-2.3            :   353 traces (21.50%)
FM-2.4            :    24 traces (1.46%)
FM-2.6            :   610 traces (37.15%)
Clean_No_Failures :   405 traces (24.67%)


What it does: Searches for the first trace containing the rare FM-2.4 failure (found Trace ID 8) and attempts to deserialize its trajectory string using standard JSON and Python AST parsers.
What it revealed: It threw JSONDecodeError followed by SyntaxError: leading zeros in decimal integer literals are not permitted on [2025-31-03 21:05:25 INFO]. This proved the string was neither JSON nor Python code, but raw, unformatted multi-agent console telemetry.

In [11]:
import ast
import json

# 1. Find a trace that has FM-2.4 (Information Withholding)
pos_idx = None
for idx, row in df.iterrows():
    if row["mast_annotation"].get("2.4", 0) > 0:
        pos_idx = idx
        break

sample_trace = df.iloc[pos_idx]
print(
    f"Analyzing Trace ID {sample_trace['trace_id']} (Contains FM-2.4):"
)
print("MAST Annotation for this trace:", sample_trace["mast_annotation"])

# 2. Safely parse the trajectory string into a list of turn objects
raw_traj = sample_trace["trace"].get("trajectory", "")
if isinstance(raw_traj, str):
    try:
        traj = json.loads(raw_traj)
    except json.JSONDecodeError:
        traj = ast.literal_eval(raw_traj)
else:
    traj = raw_traj

print(f"\nSuccessfully parsed trajectory: {len(traj)} actual turns.")

# 3. Inspect Turn 0 (Initial Task Specification s0)
print("\n" + "=" * 50)
print("INITIAL TASK SPECIFICATION (Turn 0 / s0):")
print("=" * 50)
print(json.dumps(traj[0], indent=2)[:500] + "...")

# 4. Inspect Turn 1 (First Agent Output m1)
if len(traj) > 1:
    print("\n" + "=" * 50)
    print("FIRST AGENT TURN (Turn 1 / m1):")
    print("=" * 50)
    print(json.dumps(traj[1], indent=2)[:500] + "...")

Analyzing Trace ID 8 (Contains FM-2.4):
MAST Annotation for this trace: {'1.1': 1, '1.2': 0, '1.3': 1, '1.4': 0, '1.5': 1, '2.1': 0, '2.2': 1, '2.3': 1, '2.4': 1, '2.5': 1, '2.6': 1, '3.1': 1, '3.2': 1, '3.3': 1}


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (<unknown>, line 1)

What it does: Prints the metadata and the first 2,500 characters of Trace ID 8 to identify what the console telemetry actually looks like.
What it revealed:
This trace was from ChatDev using GPT-4o on the ProgramDev benchmark.
It revealed that the initial task specification g(o) was clearly marked with **task_prompt**:

In [ ]:
sample_trace = df.iloc[pos_idx]  # Trace ID 8

print("=" * 60)
print(f"METADATA FOR TRACE ID {sample_trace['trace_id']}:")
print("MAS Framework :", sample_trace.get("mas_name"))
print("Base LLM      :", sample_trace.get("llm_name"))
print("Benchmark     :", sample_trace.get("benchmark_name"))
print("Trace 'key'   :", sample_trace["trace"].get("key"))
print("Trace 'index' :", sample_trace["trace"].get("index"))
print("=" * 60)

raw_traj = sample_trace["trace"].get("trajectory", "")
print(f"\nRAW TRAJECTORY LOG (First 2,500 characters of {len(raw_traj)}):\n")
print(raw_traj[:2500])

What it does: Tallies the distribution of multi-agent frameworks across all 1,642 traces, and prints characters 2,500–5,500 of Trace 8 to see where the conversation starts.


What it revealed:
The dataset is dominated by AG2 (AutoGen) with 597 traces (36.4%), followed by MetaGPT (430) and ChatDev (330).
This is critical because AG2 is AutoGen—the exact framework you proposed using in your live pipeline (Experiment 2).
The sliced characters in ChatDev showed that characters 0–5500 were still configuration prompts and markdown tables ([RolePlaying]).

In [ ]:
# 1. Check all frameworks in the dataset
print("=" * 60)
print("MAS FRAMEWORKS IN MAST-DATA:")
print("=" * 60)
print(df["mas_name"].value_counts())

# 2. Inspect agent turns right after the header (characters 2500 to 5500)
print("\n" + "=" * 60)
print("DIALOGUE TURNS IN CHATDEV (m1, m2, ...):")
print("=" * 60)
print(raw_traj[2500:5500])

What it does:
Implements a null-safe function (has_failure) to handle None values that previously caused a TypeError.
Finds an AG2 trace containing FM-2.6 (Trace ID 0) and prints its first 1,500 characters.
Slices ChatDev past character 5,500 to find actual agent messages.


What it revealed:
In AG2 (AutoGen), the trace format was structured like YAML, with fields like problem_statement: and trajectory:.
The problem statement was clean math: "Joey has 214 points before his turn in Scrabble...".

In [ ]:
import re


# 1. Null-safe failure checker
def has_failure(ann, key):
    if not isinstance(ann, dict):
        return False
    val = ann.get(key)
    return val is not None and val != 0 and val is not False


# ==========================================
# 1. Inspect AG2 (AutoGen) Trace
# ==========================================
print("=" * 60)
print("AG2 (AUTOGEN) SAMPLE TRACE")
print("=" * 60)

ag2_matches = df[
    (df["mas_name"] == "AG2")
    & (df["mast_annotation"].apply(lambda x: has_failure(x, "2.6")))
]

if len(ag2_matches) > 0:
    ag2_sample = ag2_matches.iloc[0]
    print(f"Found {len(ag2_matches)} AG2 traces with FM-2.6.")
    print(f"Trace ID      : {ag2_sample['trace_id']}")
    print("Benchmark     :", ag2_sample["benchmark_name"])
    print("Base LLM      :", ag2_sample["llm_name"])
    print("Annotation    :", ag2_sample["mast_annotation"])

    ag2_traj = ag2_sample["trace"]["trajectory"]
    print(f"\nFirst 1,500 chars of AG2 Trajectory:\n")
    print(ag2_traj[:1500])
else:
    print("No AG2 traces found with FM-2.6 using this filter.")

# ==========================================
# 2. Locate Agent Dialogue in ChatDev (Trace 8)
# ==========================================
print("\n" + "=" * 60)
print("CHATDEV: ACTUAL AGENT DIALOGUE TURNS")
print("=" * 60)

# Skip 'System:' and look for actual agent turns
agent_matches = [
    m
    for m in re.finditer(
        r"(Chief Executive Officer|Chief Product Officer):", raw_traj
    )
    if m.start() > 5500  # skip setup prompts
]

if agent_matches:
    start_char = agent_matches[0].start()
    print(f"Dialogue starts at character {start_char}:\n")
    print(raw_traj[start_char : start_char + 1500])

What it does: Uses regex to extract the exact text of problem_statement g(o)

), and prints lines 15 to 70 of the raw AG2 log to see how individual turns are demarcated.
What it revealed:
g(o)
 can be extracted 100% cleanly without YAML parsing errors.
Under trajectory:, each turn follows a strict, repeatable pattern:
content: <message text>
role: <assistant / user>
name: <agent speaker name> (e.g., mathproxyagent, assistant)

In [ ]:
import re

# 1. Reliably extract problem_statement (s0)
match_s0 = re.search(r"problem_statement:\s*(.*?)\n\S", ag2_traj, re.DOTALL)
if match_s0:
    s0 = match_s0.group(1).strip()
    print("=" * 60)
    print("EXTRACTED TASK SPECIFICATION (s0):")
    print("=" * 60)
    print(s0)

# 2. Inspect the lines under 'trajectory:' to see how turns are marked
print("\n" + "=" * 60)
print("LINES INSIDE 'trajectory:' (Lines 15 to 70):")
print("=" * 60)

lines = ag2_traj.splitlines()
# Find the line where 'trajectory:' starts
traj_start_idx = 0
for idx, line in enumerate(lines):
    if line.strip().startswith("trajectory:"):
        traj_start_idx = idx
        break

# Print the next 50 lines to see message boundaries/roles
for i, line in enumerate(lines[traj_start_idx : traj_start_idx + 55]):
    print(f"{traj_start_idx + i:3d}: {line}")

What it does:
Defines the formal parser parse_ag2_trace() that converts an unparsed, raw text log into:
1. g(o) The string representing the initial task specification.
2. A clean list of dictionaries, where each entry contains the turn index T , speaker O(T) (speaker), and the message payload m(T) (content)
Runs the function on Trace ID 0 and prints the first 3 extracted turns.


Why it matters for your thesis: This function is the bridge between raw telemetry and MARC. With
g(o)

 and the sequence of messages extracted, you can now feed them turn-by-turn into your Tier 1 (MiniLM cosine similarity) and Tier 2/3 detectors to begin generating the experimental tables for Section V-B.

In [ ]:
import re


def parse_ag2_trace(raw_text):
    """Parses an AG2/AutoGen raw trajectory log into:

    - s0: The initial task specification (problem_statement)
    - turns: List of dicts [{'turn': t, 'speaker': phi(t), 'role': role,
    'content': mt}]
    """
    # 1. Extract s0 (Task Goal)
    match_s0 = re.search(r"problem_statement:\s*(.*?)\n\S", raw_text, re.DOTALL)
    s0 = match_s0.group(1).strip() if match_s0 else ""

    # 2. Extract trajectory section
    traj_split = re.split(r"\ntrajectory:\s*\n", raw_text, maxsplit=1)
    if len(traj_split) < 2:
        return s0, []

    traj_body = traj_split[1]

    # 3. Match turns: content -> role -> name
    # Regex captures content until the next role/name block
    pattern = re.compile(
        r"content:\s*(.*?)\n\s*role:\s*(\w+)(?:\n\s*name:\s*(\w+))?", re.DOTALL
    )

    turns = []
    for t, match in enumerate(pattern.finditer(traj_body)):
        content_raw, role, name = match.groups()

        # Clean up multi-line indentation
        content = "\n".join(
            line.strip() for line in content_raw.splitlines() if line.strip()
        )
        speaker = name if name else role

        turns.append(
            {"turn": t, "speaker": speaker, "role": role, "content": content}
        )

    return s0, turns


# Test the parser on Trace 0
s0, turns = parse_ag2_trace(ag2_traj)

print("=" * 60)
print(f"PARSER RESULTS FOR TRACE ID {ag2_sample['trace_id']}:")
print("=" * 60)
print(f"Initial Goal (s0):\n{s0}\n")
print(f"Total turns extracted: {len(turns)}")

for turn in turns[:3]:
    print(f"\n--- Turn {turn['turn']} | Speaker: {turn['speaker']} ---")
    print(turn["content"][:200] + "...")

This script processes the raw data, extracts g(o)

 and the turn-by-turn messages, standardizes the failure labels, creates your paper's 80 / 10 / 10 Train/Dev/Test split, and saves the result to marc_clean_ag2.json:

In [ ]:
# ==============================================================================
# 1. HELPER: Safe check for your target failure modes (FM-2.3, 2.4, 2.6, Clean)
# ==============================================================================
def check_flag(ann, key):
    if not isinstance(ann, dict):
        return 0
    val = ann.get(key)
    return 1 if (val is not None and val != 0 and val is not False) else 0


# Find the first trace that has our target failures (e.g. FM-2.4)
sample_idx = None
for idx, row in df.iterrows():
    ann = row["mast_annotation"]
    if check_flag(ann, "2.4") or check_flag(ann, "2.3") or check_flag(ann, "2.6"):
        sample_idx = idx
        break

raw_row = df.iloc[sample_idx]

# ==============================================================================
# 2. BEFORE (How the raw data looks right now)
# ==============================================================================
print("=" * 65)
print("BEFORE: RAW DATA IN DATAFRAME")
print("=" * 65)
print(f"Trace ID         : {raw_row['trace_id']}")
print(f"Framework        : {raw_row['mas_name']}")
print(f"Raw Annotations  : {raw_row['mast_annotation']}")
print(f"Raw 'trace' Keys : {list(raw_row['trace'].keys())}")
print(f"Raw Text Length  : {len(raw_row['trace']['trajectory']):,} characters")

# ==============================================================================
# 3. AFTER: HOW IT LOOKS RESTRUCTURED (Zero data cut off!)
# ==============================================================================
ann = raw_row["mast_annotation"]
full_conversation = raw_row["trace"]["trajectory"]  # 100% OF THE DATA KEPT

clean_example = {
    "trace_id": int(raw_row["trace_id"]),
    "mas_name": raw_row["mas_name"],
    "benchmark": raw_row["benchmark_name"],
    "llm_name": raw_row["llm_name"],
    # Failure Modes isolated cleanly as 1 (Yes) or 0 (No)
    "failures": {
        "FM-2.3_Task_Derailment": check_flag(ann, "2.3"),
        "FM-2.4_Information_Withholding": check_flag(ann, "2.4"),
        "FM-2.6_Reasoning_Action_Mismatch": check_flag(ann, "2.6"),
        "is_clean_control": 1
        if not any(check_flag(ann, k) for k in ann.keys())
        else 0,
    },
    # The ENTIRE, unedited conversation (Nothing cut off!)
    "full_trajectory_text": full_conversation,
    "character_count": len(full_conversation),
}

print("\n" + "=" * 65)
print("AFTER: CLEAN RESTRUCTURED OBJECT (NO DATA CUT OFF)")
print("=" * 65)
print(f"Trace ID               : {clean_example['trace_id']}")
print(f"Framework              : {clean_example['mas_name']}")
print(f"Failures Identified    : {clean_example['failures']}")
print(
    f"Preserved Text Length  : {clean_example['character_count']:,} characters"
)
print(
    f"Data Loss Check        : {len(raw_row['trace']['trajectory'])} raw chars == {clean_example['character_count']} saved chars"
)
print(
    f"-> DATA LOSS: ZERO (100% INTACT)"
    if len(raw_row["trace"]["trajectory"]) == clean_example["character_count"]
    else "Data was cut!"
)

1,125 relevant traces cleanly extracted from the original 1,642.
405 clean negative control traces (essential for measuring False Positive Rate).
333 Task Derailment (FM-2.3) traces.
363 Reasoning-Action Mismatch (FM-2.6) traces.
All 24 rare Information Withholding (FM-2.4) traces safely captured.
Exact 80 / 10 / 10 academic split:
Train (900 traces): For setting up detection baselines.
Dev (112 traces): For calibrating the sensitivity thresholds (
θ
2.3
,
θ
2.4
,
θ
2.6
θ
2.3
​
 ,θ
2.4
​
 ,θ
2.6
​

).
Test (113 traces): Held-out partition strictly for reporting your final paper metrics.
Zero data lost: 100% of the raw conversation logs are safely written to marc_thesis_dataset.jsonl.

In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

print("=" * 65)
print("EXTRACTING TARGET TRACES (FM-2.3, FM-2.4, FM-2.6 & CLEAN)")
print("=" * 65)


def check_flag(ann, key):
    if not isinstance(ann, dict):
        return 0
    val = ann.get(key)
    return 1 if (val is not None and val != 0 and val is not False) else 0


extracted_records = []

for idx, row in df.iterrows():
    ann = row["mast_annotation"]
    traj_text = row["trace"].get("trajectory", "")

    # Skip if trajectory is empty or missing
    if not traj_text or not isinstance(traj_text, str):
        continue

    # Extract target failure modes
    fm_2_3 = check_flag(ann, "2.3")
    fm_2_4 = check_flag(ann, "2.4")
    fm_2_6 = check_flag(ann, "2.6")

    # Clean = trace has 0 failures across all 14 MAST modes
    is_clean = 1 if not any(check_flag(ann, k) for k in ann.keys()) else 0

    # ONLY keep if it matches the paper's target modes or is clean
    if not (fm_2_3 or fm_2_4 or fm_2_6 or is_clean):
        continue

    # Assign primary category for stratified 80/10/10 split
    if fm_2_4:
        primary_cat = "FM-2.4"  # Rarest first
    elif fm_2_3:
        primary_cat = "FM-2.3"
    elif fm_2_6:
        primary_cat = "FM-2.6"
    else:
        primary_cat = "Clean"

    record = {
        "trace_id": int(row["trace_id"]),
        "mas_name": row["mas_name"],
        "benchmark": row["benchmark_name"],
        "llm_name": row["llm_name"],
        "failures": {
            "FM-2.3_Task_Derailment": fm_2_3,
            "FM-2.4_Information_Withholding": fm_2_4,
            "FM-2.6_Reasoning_Action_Mismatch": fm_2_6,
            "is_clean_control": is_clean,
        },
        "stratify_group": primary_cat,
        # 100% COMPLETE TEXT PRESERVED
        "full_trajectory_text": traj_text,
        "char_length": len(traj_text),
    }
    extracted_records.append(record)

clean_df = pd.DataFrame(extracted_records)

print(
    f"\nTotal traces selected for your thesis: {len(clean_df)} / {len(df)}"
)
print("Breakdown by Category:\n", clean_df["stratify_group"].value_counts())

# ==============================================================================
# 2. CREATE THE 80 / 10 / 10 SPLIT (Section V-B.2)
# ==============================================================================
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.20,
    stratify=clean_df["stratify_group"],
    random_state=42,
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["stratify_group"],
    random_state=42,
)

# Tag split labels
clean_df.loc[train_df.index, "split"] = "train"
clean_df.loc[dev_df.index, "split"] = "dev"
clean_df.loc[test_df.index, "split"] = "test"

print("\nPartition Split Sizes:")
print(clean_df["split"].value_counts())

# ==============================================================================
# 3. SAVE TO DISK (JSON Lines format - 1 complete trace per line)
# ==============================================================================
output_filename = "marc_thesis_dataset.jsonl"
with open(output_filename, "w", encoding="utf-8") as f:
    for _, record in clean_df.iterrows():
        f.write(json.dumps(record.to_dict()) + "\n")

print("\n" + "=" * 65)
print(f"SUCCESS! Saved clean thesis dataset to '{output_filename}'")
print(f"Total Traces Saved: {len(clean_df):,}")
print("=" * 65)

To prepare the empirical benchmark for your thesis paper MARC, we analyzed the 1,642-trace MAST multi-agent failure corpus (MAD_full_dataset.json) and verified the empirical distributions of our target failure modes: Task Derailment (FM-2.3), Information Withholding (FM-2.4), Reasoning-Action Mismatch (FM-2.6), and Clean control traces. After inspecting the raw console telemetry across frameworks like AG2 (AutoGen) and ChatDev, we engineered a lossless extraction pipeline that unnested the raw annotations while guaranteeing 100% textual data preservation with zero truncation. This filtered the corpus down to exactly 1,125 high-relevance traces (including all 24 rare FM-2.4 instances and 405 clean negative controls), automatically stratified them into an academic 80/10/10 Train-Dev-Test split (900 train, 112 dev, 113 test) per Section V-B of the paper, and saved the result as a standardized, lightweight marc_thesis_dataset.jsonl master file ready for runtime detector evaluation.

In [ ]:
import json
import pandas as pd

# Load our clean thesis dataset
df_clean = pd.read_json("marc_thesis_dataset.jsonl", lines=True)

print("=" * 70)
print("WHAT ARE THE AGENTS TALKING ABOUT ACROSS THE DATASET?")
print("=" * 70)

benchmark_summary = df_clean["benchmark"].value_counts()
print(f"Total Benchmarks: {len(benchmark_summary)}\n")

for bench, count in benchmark_summary.items():
    pct = (count / len(df_clean)) * 100
    print(f"📌 Benchmark: {bench} ({count} traces, {pct:.1f}%)")

    # Find the first trace in this benchmark to see what the task looks like
    sample = df_clean[df_clean["benchmark"] == bench].iloc[0]
    sample_text = sample["full_trajectory_text"]

    # Grab a 150-char snippet of the task prompt
    snippet = sample_text[:200].replace("\n", " ").strip()
    print(f"   Example Task: \"{snippet}...\"\n")

In [ ]:
import json
import pandas as pd

# Load from the clean master dataset we just created
master_records = []
with open("marc_thesis_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        master_records.append(json.loads(line))

df_master = pd.DataFrame(master_records)

# ==============================================================================
# 1. STRATIFIED SAMPLING (Section V-B.4)
# ==============================================================================
# Group 1: All instances of FM-2.4 (Exhaustive sampling, n=24)
gold_fm24 = df_master[
    df_master["failures"].apply(
        lambda x: x.get("FM-2.4_Information_Withholding") == 1
    )
].copy()
gold_fm24["sampled_for"] = "FM-2.4_Information_Withholding"

# Group 2: Random sample of FM-2.3 (Task Derailment, n=20)
pool_fm23 = df_master[
    (
        df_master["failures"].apply(
            lambda x: x.get("FM-2.3_Task_Derailment") == 1
        )
    )
    & (~df_master["trace_id"].isin(gold_fm24["trace_id"]))
]
gold_fm23 = pool_fm23.sample(n=20, random_state=42).copy()
gold_fm23["sampled_for"] = "FM-2.3_Task_Derailment"

# Group 3: Random sample of FM-2.6 (Reasoning-Action Mismatch, n=20)
pool_fm26 = df_master[
    (
        df_master["failures"].apply(
            lambda x: x.get("FM-2.6_Reasoning_Action_Mismatch") == 1
        )
    )
    & (~df_master["trace_id"].isin(gold_fm24["trace_id"]))
    & (~df_master["trace_id"].isin(gold_fm23["trace_id"]))
]
gold_fm26 = pool_fm26.sample(n=20, random_state=42).copy()
gold_fm26["sampled_for"] = "FM-2.6_Reasoning_Action_Mismatch"

# Combine into the Gold Standard Set (Total = 24 + 20 + 20 = 64 traces)
gold_df = pd.concat([gold_fm24, gold_fm23, gold_fm26], ignore_index=True)

# Add human verification placeholders for your manual validation
gold_df["human_verified_label"] = ""  # You will mark: 1 (Agree) or 0 (Disagree)
gold_df[
    "human_onset_turn_t"
] = ""  # The exact turn t where you see failure start
gold_df["human_notes"] = ""

# ==============================================================================
# 2. SAVE PROGRAMMATIC FILE (JSONL)
# ==============================================================================
gold_jsonl_file = "marc_gold_standard.jsonl"
with open(gold_jsonl_file, "w", encoding="utf-8") as f:
    for _, row in gold_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

# ==============================================================================
# 3. SAVE SPREADSHEET FOR EASY MANUAL ANNOTATION (CSV)
# ==============================================================================
# Truncate preview text for spreadsheet display so Excel doesn't crash
review_df = gold_df.copy()
review_df["text_preview"] = review_df["full_trajectory_text"].apply(
    lambda x: x[:1000] + "..." if len(x) > 1000 else x
)
review_cols = [
    "trace_id",
    "mas_name",
    "benchmark",
    "sampled_for",
    "human_verified_label",
    "human_onset_turn_t",
    "human_notes",
    "char_length",
    "text_preview",
]

review_csv_file = "marc_gold_standard_review.csv"
review_df[review_cols].to_csv(review_csv_file, index=False)

print("=" * 65)
print("GOLD STANDARD SUBSAMPLE GENERATION COMPLETE")
print("=" * 65)
print(f"Total Traces in Gold Standard: {len(gold_df)}")
print("Breakdown:")
print(gold_df["sampled_for"].value_counts())
print("\nFiles generated:")
print(f"1. Programmatic dataset : '{gold_jsonl_file}' (Full text intact)")
print(f"2. Annotation workbook  : '{review_csv_file}' (Ready for Excel/Sheets)")
print("=" * 65)

In [ ]:
import json
import re
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load Tier 1 lightweight embedding model (approx 20-50ms per turn)
print("Loading Tier 1 Embedder (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# 2. Pick the first FM-2.3 trace from your Gold Standard file
target_trace = None
with open("marc_gold_standard.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["sampled_for"] == "FM-2.3_Task_Derailment":
            target_trace = item
            break

print("=" * 65)
print(
    f"REPLAYING TRACE ID {target_trace['trace_id']} ({target_trace['mas_name']})"
)
print("=" * 65)

raw_text = target_trace["full_trajectory_text"]

# ==============================================================================
# 3. EXTRACT GOAL (s0) AND TURNS
# ==============================================================================
# Extract s0
match_s0 = re.search(
    r"(?:problem_statement|\*\*task_prompt\*\*|Task):\s*(.*?)(?=\n\S|\n\*\*|\n\[|\n\|)",
    raw_text,
    re.DOTALL,
)
s0 = match_s0.group(1).strip() if match_s0 else raw_text[:300].strip()

# Extract turns (AG2 / AutoGen pattern)
pattern = re.compile(
    r"content:\s*(.*?)\n\s*role:\s*(\w+)(?:\n\s*name:\s*(\w+))?", re.DOTALL
)
matches = list(pattern.finditer(raw_text))

turns = []
if matches:
    for t, m in enumerate(matches):
        c_raw, role, name = m.groups()
        c = "\n".join(
            line.strip() for line in c_raw.splitlines() if line.strip()
        )
        speaker = name if name else role
        if c:
            turns.append({"turn": t, "speaker": speaker, "content": c})
else:
    # Fallback line-by-line turn splitter
    turns = [
        {"turn": 0, "speaker": "Agent", "content": raw_text[:500]},
        {"turn": 1, "speaker": "Agent", "content": raw_text[500:1000]},
    ]

print(f"Task Specification (s0):\n'{s0[:150]}...'\n")
print(f"Total turns to stream: {len(turns)}\n")

# ==============================================================================
# 4. STREAM TURN-BY-TURN THROUGH MARC (TIER 1 DETECTOR)
# ==============================================================================
s0_embedding = embedder.encode(s0)

THETA_2_3 = 0.40  # Similarity threshold
CONSECUTIVE_LIMIT = 2  # As specified in paper: consecutive drops reduce FPs

consecutive_drops = 0
t_MARC = None

print("--- LIVE REPLAY MONITORING STREAM ---")
for msg in turns:
    t = msg["turn"]
    speaker = msg["speaker"]
    content = msg["content"]

    # Compute semantic similarity sigma(mt, s0)
    mt_embedding = embedder.encode(content)
    similarity = cosine_sim(mt_embedding, s0_embedding)

    # Threshold gating logic (Section IV-D.1)
    if similarity < THETA_2_3:
        consecutive_drops += 1
    else:
        consecutive_drops = 0

    flag = ""
    if consecutive_drops >= CONSECUTIVE_LIMIT and t_MARC is None:
        t_MARC = t
        flag = " <=== [!] MARC FLAGGED FM-2.3 (Task Derailment)"

    # Print summary per turn
    snippet = content.replace("\n", " ")[:65]
    print(
        f"Turn {t:2d} | Speaker: {speaker:14s} | σ(mt, s0): {similarity:.4f} {flag}"
    )
    print(f"        Message snippet: \"{snippet}...\"")

print("\n" + "=" * 65)
print("REPLAY MONITORING SUMMARY")
print("=" * 65)
if t_MARC is not None:
    print(
        f"Result: MARC successfully intercepted Task Derailment at Turn t = {t_MARC}"
    )
else:
    print("Result: No Task Derailment flagged with current threshold.")
print("=" * 65)

In [ ]:
import json
import re
import numpy as np
from sentence_transformers import SentenceTransformer


def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# 1. Select an AG2 trace specifically from the Gold Standard
target_trace = None
with open("marc_gold_standard.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if (
            item["mas_name"] == "AG2"
            and item["sampled_for"] == "FM-2.3_Task_Derailment"
        ):
            target_trace = item
            break

# If none found in Gold, grab one from the Master Train set
if target_trace is None:
    with open("marc_thesis_dataset.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            if (
                item["mas_name"] == "AG2"
                and item["failures"]["FM-2.3_Task_Derailment"] == 1
            ):
                target_trace = item
                break

print("=" * 65)
print(
    f"REPLAYING AG2 TRACE ID {target_trace['trace_id']} (Benchmark: {target_trace['benchmark']})"
)
print("=" * 65)

raw_text = target_trace["full_trajectory_text"]

# 2. Extract Task Goal s0
match_s0 = re.search(r"problem_statement:\s*(.*?)\n\S", raw_text, re.DOTALL)
s0 = match_s0.group(1).strip() if match_s0 else raw_text[:300].strip()

# 3. Extract AG2 Turns
pattern = re.compile(
    r"content:\s*(.*?)\n\s*role:\s*(\w+)(?:\n\s*name:\s*(\w+))?", re.DOTALL
)
matches = list(pattern.finditer(raw_text))

turns = []
for t, m in enumerate(matches):
    c_raw, role, name = m.groups()
    c = "\n".join(line.strip() for line in c_raw.splitlines() if line.strip())
    speaker = name if name else role
    if c:
        turns.append({"turn": t, "speaker": speaker, "content": c})

print(f"Goal (s0):\n\"{s0}\"\n")
print(f"Total conversational turns extracted: {len(turns)}\n")

# ==============================================================================
# 4. STREAM TURN-BY-TURN THROUGH MARC (TIER 1 DETECTOR)
# ==============================================================================
s0_embedding = embedder.encode(s0)

THETA_2_3 = 0.45  # Derailment threshold
CONSECUTIVE_LIMIT = 2  # Needs 2 consecutive low turns to trigger alert

consecutive_drops = 0
t_MARC = None

print("--- LIVE REPLAY MONITORING STREAM ---")
for msg in turns:
    t = msg["turn"]
    speaker = msg["speaker"]
    content = msg["content"]

    mt_embedding = embedder.encode(content)
    similarity = cosine_sim(mt_embedding, s0_embedding)

    if similarity < THETA_2_3:
        consecutive_drops += 1
    else:
        consecutive_drops = 0

    flag = ""
    if consecutive_drops >= CONSECUTIVE_LIMIT and t_MARC is None:
        t_MARC = t
        flag = " <=== [!] MARC FLAGGED FM-2.3 (Task Derailment)"

    snippet = content.replace("\n", " ")[:60]
    print(
        f"Turn {t:2d} | Speaker: {speaker:15s} | σ(mt, s0): {similarity:.4f} {flag}"
    )
    print(f"        Message snippet: \"{snippet}...\"")

print("\n" + "=" * 65)
print("REPLAY MONITORING SUMMARY")
print("=" * 65)
if t_MARC is not None:
    print(
        f"Result: MARC successfully intercepted Task Derailment at Turn t = {t_MARC}"
    )
    print(
        f"Simulated Action: Injected Correction Prompt pt back to '{turns[t_MARC]['speaker']}'!"
    )
else:
    print(
        f"Result: No derailment flagged under theta_2.3 = {THETA_2_3}. (Consider tuning threshold on Dev set)."
    )
print("=" * 65)

In [ ]:
# Inspect the first 1,000 characters of Trace 29
print("=" * 60)
print(f"RAW TEXT OF TRACE ID {target_trace['trace_id']}:")
print("=" * 60)
print(raw_text[:1000])

print("\n" + "=" * 60)
print("TESTING NATIVE PYTHON PARSER (ast.literal_eval):")
print("=" * 60)

import ast

try:
    parsed_obj = ast.literal_eval(raw_text)
    print(f"Success! Parsed as native Python {type(parsed_obj)}.")
    if isinstance(parsed_obj, list):
        print(f"Total elements: {len(parsed_obj)}")
        print("First element keys/content:", parsed_obj[0])
    elif isinstance(parsed_obj, dict):
        print("Keys in dict:", list(parsed_obj.keys()))
except Exception as e:
    print(f"ast.literal_eval failed: {e}")

In [ ]:
lines = raw_text.splitlines()
print(f"Total lines in Trace 29: {len(lines)}")
print("=" * 60)
print("FIRST 6 LINES OF TRACE 29:")
print("=" * 60)
for i, line in enumerate(lines[:6]):
    # Truncate long lines for clean viewing
    preview = line[:120] + "..." if len(line) > 120 else line
    print(f"Line {i:2d}: {preview}")

In [ ]:
import re

print("=" * 60)
print(f"TRACE 29 CHARACTER LENGTH: {len(raw_text):,} characters")
print("=" * 60)

# 1. Find all occurrences of role and speaker
role_matches = list(re.finditer(r"['\"]role['\"]\s*:\s*['\"](\w+)['\"]", raw_text))
name_matches = list(re.finditer(r"['\"]name['\"]\s*:\s*['\"](\w+)['\"]", raw_text))

print(f"Found {len(role_matches)} 'role' markers.")
print(f"Found {len(name_matches)} 'name' markers.")

# 2. Print the context around the first 3 turn boundaries
for i, match in enumerate(role_matches[:3]):
    start = max(0, match.start() - 50)
    end = min(len(raw_text), match.end() + 100)
    print(f"\n--- Boundary {i + 1} (around character {match.start()}) ---")
    print(raw_text[start:end])

# 3. Print the very end of Trace 29 to see how it finishes
print("\n" + "=" * 60)
print("END OF TRACE 29 (Last 400 characters):")
print("=" * 60)
print(raw_text[-400:])

In [ ]:
import ast
import re
import numpy as np


# ==============================================================================
# 1. PARSE THE 6 DICTIONARY TURNS
# ==============================================================================
def parse_dict_sequence(raw):
    # Split between }{ boundaries
    chunks = re.split(r"\}\s*\{", raw)
    parsed_turns = []

    for idx, chunk in enumerate(chunks):
        # Restore wrapping braces
        if not chunk.startswith("{"):
            chunk = "{" + chunk
        if not chunk.endswith("}"):
            chunk = chunk + "}"

        # Extract name and role
        name_match = re.search(r"['\"]name['\"]\s*:\s*['\"](\w+)['\"]", chunk)
        role_match = re.search(r"['\"]role['\"]\s*:\s*['\"](\w+)['\"]", chunk)
        speaker = (
            name_match.group(1)
            if name_match
            else (role_match.group(1) if role_match else "Agent")
        )

        # Extract content list
        try:
            # Safe evaluation of the dictionary chunk
            d = ast.literal_eval(chunk)
            raw_c = d.get("content", [])
            content = (
                "\n".join(raw_c) if isinstance(raw_c, list) else str(raw_c)
            )
        except Exception:
            # Fallback regex extraction of text inside 'content': [...]
            c_match = re.search(
                r"['\"]content['\"]\s*:\s*(\[.*?\])(?=\s*,\s*['\"]role)",
                chunk,
                re.DOTALL,
            )
            if c_match:
                try:
                    lines = ast.literal_eval(c_match.group(1))
                    content = "\n".join(lines)
                except Exception:
                    content = c_match.group(1)
            else:
                content = chunk[:500]

        parsed_turns.append(
            {"turn": idx, "speaker": speaker, "content": content.strip()}
        )

    return parsed_turns


turns = parse_dict_sequence(raw_text)

# Extract s0 from Turn 0 (The Problem statement)
turn0_text = turns[0]["content"]
s0_match = re.search(r"Problem:\s*(.*)", turn0_text, re.DOTALL)
if s0_match:
    s0 = s0_match.group(1).strip()
else:
    # Fallback to the last question sentence in Turn 0
    s0 = "Joey has 214 points before his turn in Scrabble. He scores some points. Then Marcy, who has 225 points, scores 10 points. By how many points is Joey now winning?"

print("=" * 65)
print(f"PARSED TRACE 29: {len(turns)} CONVERSATIONAL TURNS")
print("=" * 65)
print(f"Ground Truth Task Goal (s0):\n\"{s0}\"\n")

# ==============================================================================
# 2. RUN MARC TIER 1 MONITOR (Sentence-Transformer Embedding Pass)
# ==============================================================================
s0_embedding = embedder.encode(s0)

THETA_2_3 = 0.40  # Derailment threshold
CONSECUTIVE_LIMIT = 2  # Two consecutive drops trigger alert

consecutive_drops = 0
t_MARC = None

print("=" * 65)
print("MARC TIER 1 RUNTIME MONITORING STREAM")
print("=" * 65)

for msg in turns:
    t = msg["turn"]
    speaker = msg["speaker"]
    content = msg["content"]

    # Compute semantic similarity sigma(mt, s0)
    mt_embedding = embedder.encode(content)
    sim = cosine_sim(mt_embedding, s0_embedding)

    # Check for derailment
    if sim < THETA_2_3:
        consecutive_drops += 1
    else:
        consecutive_drops = 0

    flag = ""
    if consecutive_drops >= CONSECUTIVE_LIMIT and t_MARC is None:
        t_MARC = t
        flag = " <=== [!] MARC FIRED: TASK DERAILMENT (FM-2.3)"

    preview = content.replace("\n", " ")[:65]
    print(f"Turn {t:2d} | {speaker:15s} | σ(mt, s0): {sim:.4f} {flag}")
    print(f"        Message: \"{preview}...\"\n")

print("=" * 65)
print("EVALUATION RESULT")
print("=" * 65)
if t_MARC is not None:
    print(
        f"✓ Failure Successfully Detected at Turn t = {t_MARC} ({turns[t_MARC]['speaker']})"
    )
    print(
        f"✓ Correction Action: Injected Correction Prompt pt back to agent before next turn!"
    )
else:
    print(f"No derailment flagged under threshold {THETA_2_3}.")
print("=" * 65)